# Chain-of-Thought (CoT) Study - 1. Source QA

This notebook runs Source QA using the improved **P2-cot** strategy with structured FINAL ANSWER output.

**Run this notebook FIRST before the language notebooks.**

## Environment Setup

In [ ]:
import os
import sys

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    
    print(f'Model cache directory: {DRIVE_CACHE_DIR}')
elif IN_KAGGLE:
    print('Running on Kaggle - models will be cached in /root/.cache')
else:
    print('Running locally - using default cache directories')

In [ ]:
import subprocess

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
                'transformers', 'torch', 'accelerate'], check=True)
print('Dependencies installed!')

In [ ]:
# Clone repository based on environment
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        print(f'Cloning repository to {PROJECT_ROOT}...')
        subprocess.run(['git', 'clone', 
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', 
                        PROJECT_ROOT], check=True)
        print('Clone complete!')
    else:
        print(f'Repository already exists at {PROJECT_ROOT}')
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone', 
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', 
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print('=== Downloading/Loading Qwen Model ===')
print('This may take a while on first run...\n')

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('✓ Qwen cached')

## Path Configuration

In [ ]:
# Path Configuration
ABLATION_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/prompt-ablation"
COT_DIR = f"{ABLATION_DIR}/cot"
CODE_DIR = f"{ABLATION_DIR}/code"
QG_PATH = f"{PROJECT_ROOT}/results Qwen3B baseline/biomqm/baseline/QG/qwen-3b.jsonl"

STRATEGY = "P2-cot"

# Create output directory
os.makedirs(f"{COT_DIR}/QA", exist_ok=True)

# Add code directory to path
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"COT_DIR: {COT_DIR}")
print(f"QG_PATH: {QG_PATH}")

In [ ]:
# Verify files exist
print("\n=== Verifying Files ===")

code_files = ['prompts.py', 'qa_ablation.py']

all_ok = True
for f in code_files:
    path = os.path.join(CODE_DIR, f)
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"{status} {f}")
    if not exists:
        all_ok = False

if os.path.exists(QG_PATH):
    with open(QG_PATH, 'r') as f:
        n_lines = sum(1 for _ in f)
    print(f"\n✓ QG file found ({n_lines} lines)")
else:
    print(f"\n✗ QG file NOT found: {QG_PATH}")
    all_ok = False

if all_ok:
    print("\n=== All files verified! ===")
else:
    print("\n=== WARNING: Some files missing! ===")

## Source QA - Chain-of-Thought

In [ ]:
# P2-cot Source QA
output_file = f"{COT_DIR}/QA/source-{STRATEGY}.jsonl"

cmd = [
    sys.executable, "-u",
    f"{CODE_DIR}/qa_ablation.py",
    "--strategy", STRATEGY,
    "--mode", "source",
    "--qg_input_path", QG_PATH,
    "--output_path", output_file
]

print(f"Running Source QA with {STRATEGY}...")
print(f"Output: {output_file}")
subprocess.run(cmd, check=True)
print(f"✓ Source QA {STRATEGY} complete!")

## Git Push

In [ ]:
os.chdir(PROJECT_ROOT)
subprocess.run(['git', 'config', '--global', 'user.email', 'simone@example.com'])
subprocess.run(['git', 'config', '--global', 'user.name', 'Simone'])
subprocess.run(['git', 'add', '-A'])
subprocess.run(['git', 'commit', '-m', 'Add CoT source QA results'])
subprocess.run(['git', 'push', 'origin', 'main'])
print("✓ Push complete!")

## Summary

In [ ]:
print("\n" + "="*60)
print("SOURCE QA COMPLETE - CHAIN-OF-THOUGHT")
print("="*60)
output_file = f"{COT_DIR}/QA/source-{STRATEGY}.jsonl"
if os.path.exists(output_file):
    with open(output_file, 'r') as f:
        n_lines = sum(1 for _ in f)
    print(f"✓ {STRATEGY}: {n_lines} lines")
else:
    print(f"✗ {STRATEGY}: File not found")
print("\nNow run the language notebooks (2-6) for BT QA.")